In [4]:
import pandas as pd

# Load the 2008 Beneficiary Summary file
beneficiary_2008 = pd.read_csv("dataSets/DE1_0_2008_Beneficiary_Summary_File_Sample_1.csv")

# Load the PDE (Prescription Drug Events) file
pde = pd.read_csv(
    "dataSets/DE1_0_2008_to_2010_Prescription_Drug_Events_Sample_1.csv",
    dtype={"PROD_SRVC_ID": str}
)

In [5]:
print("Beneficiary 2008 shape:", beneficiary_2008.shape)
print("PDE shape:", pde.shape)

Beneficiary 2008 shape: (116352, 32)
PDE shape: (5552421, 8)


In [6]:
print("\nBeneficiary 2008 columns:")
print(list(beneficiary_2008.columns))


Beneficiary 2008 columns:
['DESYNPUF_ID', 'BENE_BIRTH_DT', 'BENE_DEATH_DT', 'BENE_SEX_IDENT_CD', 'BENE_RACE_CD', 'BENE_ESRD_IND', 'SP_STATE_CODE', 'BENE_COUNTY_CD', 'BENE_HI_CVRAGE_TOT_MONS', 'BENE_SMI_CVRAGE_TOT_MONS', 'BENE_HMO_CVRAGE_TOT_MONS', 'PLAN_CVRG_MOS_NUM', 'SP_ALZHDMTA', 'SP_CHF', 'SP_CHRNKIDN', 'SP_CNCR', 'SP_COPD', 'SP_DEPRESSN', 'SP_DIABETES', 'SP_ISCHMCHT', 'SP_OSTEOPRS', 'SP_RA_OA', 'SP_STRKETIA', 'MEDREIMB_IP', 'BENRES_IP', 'PPPYMT_IP', 'MEDREIMB_OP', 'BENRES_OP', 'PPPYMT_OP', 'MEDREIMB_CAR', 'BENRES_CAR', 'PPPYMT_CAR']


In [7]:
print("\nPDE columns:")
print(list(pde.columns))


PDE columns:
['DESYNPUF_ID', 'PDE_ID', 'SRVC_DT', 'PROD_SRVC_ID', 'QTY_DSPNSD_NUM', 'DAYS_SUPLY_NUM', 'PTNT_PAY_AMT', 'TOT_RX_CST_AMT']


In [8]:
print("Beneficiary 2008 sample rows:")
display(beneficiary_2008.head(3))

Beneficiary 2008 sample rows:


,DESYNPUF_ID,BENE_BIRTH_DT,BENE_DEATH_DT,BENE_SEX_IDENT_CD,BENE_RACE_CD,BENE_ESRD_IND,SP_STATE_CODE,BENE_COUNTY_CD,BENE_HI_CVRAGE_TOT_MONS,BENE_SMI_CVRAGE_TOT_MONS,...,SP_STRKETIA,MEDREIMB_IP,BENRES_IP,PPPYMT_IP,MEDREIMB_OP,BENRES_OP,PPPYMT_OP,MEDREIMB_CAR,BENRES_CAR,PPPYMT_CAR
0,00013D2EFD8E45D1,19230501,NaN,1,1,0,26,950,12,12,...,2,0.0,0.0,0.0,50.0,10.0,0.0,0.0,0.0,0.0
1,00016F745862898F,19430101,NaN,1,1,0,39,230,12,12,...,2,0.0,0.0,0.0,0.0,0.0,0.0,700.0,240.0,0.0
2,0001FDD721E223DC,19360901,NaN,2,1,0,39,280,12,12,...,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
print("\nPDE sample rows:")
display(pde.head(3))


PDE sample rows:


,DESYNPUF_ID,PDE_ID,SRVC_DT,PROD_SRVC_ID,QTY_DSPNSD_NUM,DAYS_SUPLY_NUM,PTNT_PAY_AMT,TOT_RX_CST_AMT
0,00013D2EFD8E45D1,233664490397622,20080103,00247037252,30.0,20,10.0,120.0
1,00013D2EFD8E45D1,233644490171972,20080105,00223039502,10.0,10,0.0,0.0
2,00013D2EFD8E45D1,233974489116848,20080109,00364724812,120.0,30,10.0,110.0


In [10]:
print("\nPart D coverage months (PLAN_CVRG_MOS_NUM) distribution:")
print(beneficiary_2008['PLAN_CVRG_MOS_NUM'].value_counts().sort_index())


Part D coverage months (PLAN_CVRG_MOS_NUM) distribution:
PLAN_CVRG_MOS_NUM
0     47297
1       462
2       390
3       412
4       422
5       471
6       554
7       568
8       755
9       837
10     1015
11     1299
12    61870
Name: count, dtype: int64


Beneficiary 2008

In [11]:
# --- Step 1: Clean the Beneficiary Summary file ---
bene = beneficiary_2008.copy()

In [12]:
# Convert date columns from YYYYMMDD number to real dates
bene['BENE_BIRTH_DT'] = pd.to_datetime(bene['BENE_BIRTH_DT'], format='%Y%m%d')
bene['BENE_DEATH_DT'] = pd.to_datetime(bene['BENE_DEATH_DT'], format='%Y%m%d')
print(bene['BENE_BIRTH_DT'].dtype)
print(bene['BENE_DEATH_DT'].dtype)

datetime64[ns]
datetime64[ns]


In [13]:
# Calculate age as of 2008 (reference year for this file)
bene['AGE'] = 2008 - bene['BENE_BIRTH_DT'].dt.year
print(bene[['AGE', 'BENE_BIRTH_DT']].head(3))

   AGE BENE_BIRTH_DT
0   85    1923-05-01
1   65    1943-01-01
2   72    1936-09-01


In [14]:
# Fix chronic condition columns: 1 = has condition, 2 = does not -> convert to 1/0
chronic_cols = ['SP_ALZHDMTA', 'SP_CHF', 'SP_CHRNKIDN', 'SP_CNCR', 'SP_COPD',
                 'SP_DEPRESSN', 'SP_DIABETES', 'SP_ISCHMCHT', 'SP_OSTEOPRS',
                 'SP_RA_OA', 'SP_STRKETIA']

for col in chronic_cols:
    bene[col] = bene[col].map({1: 1, 2: 0})

In [15]:
# Sanity check: chronic condition rates should now look like proper percentages
print("Chronic condition prevalence (%) after fix:")
print((bene[chronic_cols].mean() * 100).round(1))

Chronic condition prevalence (%) after fix:
SP_ALZHDMTA    19.3
SP_CHF         28.5
SP_CHRNKIDN    16.1
SP_CNCR         6.4
SP_COPD        13.5
SP_DEPRESSN    21.3
SP_DIABETES    37.9
SP_ISCHMCHT    42.1
SP_OSTEOPRS    17.3
SP_RA_OA       15.4
SP_STRKETIA     4.5
dtype: float64


In [16]:
# Filter to patients with full-year Part D coverage
bene_full_coverage = bene[bene['PLAN_CVRG_MOS_NUM'] == 12].copy()
print("\nPatients with full Part D coverage:", len(bene_full_coverage))


Patients with full Part D coverage: 61870


In [17]:
# Keep only the columns we actually need going forward
keep_cols = ['DESYNPUF_ID', 'AGE', 'BENE_SEX_IDENT_CD', 'BENE_RACE_CD',
             'BENE_DEATH_DT'] + chronic_cols

bene_clean = bene_full_coverage[keep_cols].copy()

print("\nCleaned Beneficiary table shape:", bene_clean.shape)
display(bene_clean.head())


Cleaned Beneficiary table shape: (61870, 16)


,DESYNPUF_ID,AGE,BENE_SEX_IDENT_CD,BENE_RACE_CD,BENE_DEATH_DT,SP_ALZHDMTA,SP_CHF,SP_CHRNKIDN,SP_CNCR,SP_COPD,SP_DEPRESSN,SP_DIABETES,SP_ISCHMCHT,SP_OSTEOPRS,SP_RA_OA,SP_STRKETIA
0,00013D2EFD8E45D1,85,1,1,NaT,0,0,0,0,0,0,0,0,0,0,0
2,0001FDD721E223DC,72,2,1,NaT,0,0,0,0,0,0,0,0,0,0,0
9,00036A21B65B0206,70,2,2,NaT,0,0,0,0,0,0,0,0,0,0,0
10,000489E7EAAD463F,74,2,1,NaT,0,0,0,0,0,1,1,1,0,0,0
12,0004F0ABD505251D,72,2,1,NaT,1,0,0,1,1,0,1,1,0,0,0


Got the Variable from 32 -> 16
- removed the coveraged for part A,B and HMO (-3)
- selecting only Part D coverage with 12 months(-1)
- 'MEDREIMB_IP', 'BENRES_IP', 'PPPYMT_IP', 'MEDREIMB_OP', 'BENRES_OP', 
'PPPYMT_OP', 'MEDREIMB_CAR', 'BENRES_CAR', 'PPPYMT_CAR','BENE_ESRD_IND', 
'SP_STATE_CODE', 'BENE_COUNTY_CD' removed (-12)

so totally 16 columns are removed in the beneficiary 2008

In [18]:
# Check the structure/length of PROD_SRVC_ID to figure out how to truncate it sensibly
print(pde['PROD_SRVC_ID'].str.len().value_counts())
print(pde['PROD_SRVC_ID'].head(10).tolist())

PROD_SRVC_ID
11    5552398
5          23
Name: count, dtype: int64
['00247037252', '00223039502', '00364724812', '00179180672', '58016005300', '53650001801', '00008032506', '58016081825', '51129143701', '67668013231']


In [19]:
# --- Rebuild PDE-side pipeline (NDC9 grouping -> maintenance pairs -> PDC) ---

# Convert SRVC_DT to real dates (skip if already done earlier in your session)
pde['SRVC_DT'] = pd.to_datetime(pde['SRVC_DT'], format='%Y%m%d')

# Drug-specific grouping key: labeler + product (drop package segment)
pde['NDC9'] = pde['PROD_SRVC_ID'].str[:9]

# Identify (patient, drug) pairs with at least 2 fills -- our "maintenance therapy" cohort
fills_per_pair_9 = pde.groupby(['DESYNPUF_ID', 'NDC9']).size().reset_index(name='fill_count')
maintenance_pairs_9 = fills_per_pair_9[fills_per_pair_9['fill_count'] >= 2]

print("Maintenance pairs at NDC9, 2+ fills:", len(maintenance_pairs_9))
print("Unique patients covered:", maintenance_pairs_9['DESYNPUF_ID'].nunique())

# Filter PDE down to only these maintenance pairs
pde_pdc9 = pde.merge(maintenance_pairs_9[['DESYNPUF_ID', 'NDC9']], on=['DESYNPUF_ID', 'NDC9'], how='inner')

# Calculate PDC (Proportion of Days Covered) per (patient, drug)
pdc_calc9 = pde_pdc9.groupby(['DESYNPUF_ID', 'NDC9']).agg(
    first_fill=('SRVC_DT', 'min'),
    last_fill=('SRVC_DT', 'max'),
    total_days_supplied=('DAYS_SUPLY_NUM', 'sum'),
    num_fills=('SRVC_DT', 'count')
).reset_index()

pdc_calc9['last_fill_end'] = pdc_calc9['last_fill'] + pd.to_timedelta(
    pde_pdc9.groupby(['DESYNPUF_ID', 'NDC9'])['DAYS_SUPLY_NUM'].last().values, unit='D'
)

pdc_calc9['observation_days'] = (pdc_calc9['last_fill_end'] - pdc_calc9['first_fill']).dt.days
pdc_calc9['PDC'] = (pdc_calc9['total_days_supplied'] / pdc_calc9['observation_days']).clip(upper=1.0)

print("\nPDC distribution at NDC9 (drug-specific):")
print(pdc_calc9['PDC'].describe())

Maintenance pairs at NDC9, 2+ fills: 28526
Unique patients covered: 20224

PDC distribution at NDC9 (drug-specific):
count    28526.000000
mean         0.342042
std          0.278343
min          0.000000
25%          0.134983
50%          0.235294
75%          0.456274
max          1.000000
Name: PDC, dtype: float64


In [20]:
# --- Step 6: Collapse PDC to one row per PATIENT ---

patient_pdc = pdc_calc9.groupby('DESYNPUF_ID').agg(
    avg_pdc=('PDC', 'mean'),
    min_pdc=('PDC', 'min'),          # worst-covered therapy
    num_maintenance_drugs=('NDC9', 'nunique'),
    total_fills=('num_fills', 'sum')
).reset_index()

print("Patient-level PDC summary shape:", patient_pdc.shape)
print(patient_pdc.describe())

# --- Step 7: Convert avg_pdc into risk tiers using percentile rank ---
patient_pdc['pdc_percentile'] = patient_pdc['avg_pdc'].rank(pct=True)

def tier_from_percentile(p):
    if p <= 0.33:
        return 'High Risk'
    elif p <= 0.66:
        return 'Medium Risk'
    else:
        return 'Low Risk'

patient_pdc['risk_tier'] = patient_pdc['pdc_percentile'].apply(tier_from_percentile)

print("\nRisk tier counts:")
print(patient_pdc['risk_tier'].value_counts())

display(patient_pdc.head(10))

Patient-level PDC summary shape: (20224, 5)
            avg_pdc       min_pdc  num_maintenance_drugs   total_fills
count  20224.000000  20224.000000           20224.000000  20224.000000
mean       0.343791      0.296868               1.410502      2.877324
std        0.256166      0.258203               0.736849      1.538069
min        0.000000      0.000000               1.000000      2.000000
25%        0.150678      0.117955               1.000000      2.000000
50%        0.256410      0.194805               1.000000      2.000000
75%        0.464378      0.375000               2.000000      4.000000
max        1.000000      1.000000               7.000000     17.000000

Risk tier counts:
risk_tier
Low Risk       6876
Medium Risk    6680
High Risk      6668
Name: count, dtype: int64


,DESYNPUF_ID,avg_pdc,min_pdc,num_maintenance_drugs,total_fills,pdc_percentile,risk_tier
0,00013D2EFD8E45D1,0.256556,0.207612,2,5,0.500494,Medium Risk
1,0001FDD721E223DC,0.441795,0.090090,3,6,0.732199,Low Risk
2,00036A21B65B0206,0.080537,0.080537,1,2,0.048482,High Risk
3,00108066CA1FACCE,0.099256,0.099256,1,2,0.100672,High Risk
4,0011714C14B52EEB,0.208333,0.208333,1,2,0.400242,Medium Risk
5,0011CB1FE23E91AF,0.152284,0.152284,1,2,0.254524,High Risk
6,0012AFEEC379A69D,0.483871,0.483871,1,2,0.765773,Low Risk
7,00139C345A104F72,1.000000,1.000000,1,2,0.971964,Low Risk
8,00151A878F9A2C0D,0.241935,0.241935,1,2,0.470802,Medium Risk
9,00157F1570C74E09,0.180723,0.180723,1,2,0.332723,Medium Risk


In [21]:
# Load 2009 and 2010 Beneficiary files to check overlap
beneficiary_2009 = pd.read_csv("dataSets/DE1_0_2009_Beneficiary_Summary_File_Sample_1.csv")
beneficiary_2010 = pd.read_csv("dataSets/DE1_0_2010_Beneficiary_Summary_File_Sample_1.csv")

ids_2008 = set(beneficiary_2008['DESYNPUF_ID'])
ids_2009 = set(beneficiary_2009['DESYNPUF_ID'])
ids_2010 = set(beneficiary_2010['DESYNPUF_ID'])

print("Patients in 2008:", len(ids_2008))
print("Patients in 2009:", len(ids_2009))
print("Patients in 2010:", len(ids_2010))

print("\nPatients in PDE data but NOT in 2008 Beneficiary file:")
pde_patients = set(pde['DESYNPUF_ID'])
missing_from_2008 = pde_patients - ids_2008
print(len(missing_from_2008))

print("\nOf those missing from 2008, how many appear in 2009 or 2010 instead?")
found_elsewhere = missing_from_2008 & (ids_2009 | ids_2010)
print(len(found_elsewhere))

Patients in 2008: 116352
Patients in 2009: 114538
Patients in 2010: 112754

Patients in PDE data but NOT in 2008 Beneficiary file:
0

Of those missing from 2008, how many appear in 2009 or 2010 instead?
0


In [22]:
# --- Step 8: Merge PDC/risk data with cleaned Beneficiary demographics ---

patient_features = bene_clean.merge(patient_pdc, on='DESYNPUF_ID', how='left')

print("Final merged shape:", patient_features.shape)

no_therapy_count = patient_features['avg_pdc'].isna().sum()
print(f"Patients with no maintenance therapy on record: {no_therapy_count}")

patient_features['risk_tier'] = patient_features['risk_tier'].fillna('Insufficient Data')

numeric_fill_cols = ['avg_pdc', 'min_pdc', 'num_maintenance_drugs', 'total_fills', 'pdc_percentile']
patient_features[numeric_fill_cols] = patient_features[numeric_fill_cols].fillna(-1)

print("\nFinal risk_tier distribution (including Insufficient Data):")
print(patient_features['risk_tier'].value_counts())

print("\nFinal columns:", list(patient_features.columns))
display(patient_features.head(10))

# --- Step 9: Save to CSV ---
patient_features.to_csv("dataSets/patient_features.csv", index=False)
print("\nSaved patient_features.csv successfully!")

Final merged shape: (61870, 22)
Patients with no maintenance therapy on record: 42426

Final risk_tier distribution (including Insufficient Data):
risk_tier
Insufficient Data    42426
Low Risk              6610
High Risk             6420
Medium Risk           6414
Name: count, dtype: int64

Final columns: ['DESYNPUF_ID', 'AGE', 'BENE_SEX_IDENT_CD', 'BENE_RACE_CD', 'BENE_DEATH_DT', 'SP_ALZHDMTA', 'SP_CHF', 'SP_CHRNKIDN', 'SP_CNCR', 'SP_COPD', 'SP_DEPRESSN', 'SP_DIABETES', 'SP_ISCHMCHT', 'SP_OSTEOPRS', 'SP_RA_OA', 'SP_STRKETIA', 'avg_pdc', 'min_pdc', 'num_maintenance_drugs', 'total_fills', 'pdc_percentile', 'risk_tier']


,DESYNPUF_ID,AGE,BENE_SEX_IDENT_CD,BENE_RACE_CD,BENE_DEATH_DT,SP_ALZHDMTA,SP_CHF,SP_CHRNKIDN,SP_CNCR,SP_COPD,...,SP_ISCHMCHT,SP_OSTEOPRS,SP_RA_OA,SP_STRKETIA,avg_pdc,min_pdc,num_maintenance_drugs,total_fills,pdc_percentile,risk_tier
0,00013D2EFD8E45D1,85,1,1,NaT,0,0,0,0,0,...,0,0,0,0,0.256556,0.207612,2.0,5.0,0.500494,Medium Risk
1,0001FDD721E223DC,72,2,1,NaT,0,0,0,0,0,...,0,0,0,0,0.441795,0.090090,3.0,6.0,0.732199,Low Risk
2,00036A21B65B0206,70,2,2,NaT,0,0,0,0,0,...,0,0,0,0,0.080537,0.080537,1.0,2.0,0.048482,High Risk
3,000489E7EAAD463F,74,2,1,NaT,0,0,0,0,0,...,1,0,0,0,-1.000000,-1.000000,-1.0,-1.0,-1.000000,Insufficient Data
4,0004F0ABD505251D,72,2,1,NaT,1,0,0,1,1,...,1,0,0,0,-1.000000,-1.000000,-1.0,-1.0,-1.000000,Insufficient Data
5,0007F12A492FD25D,89,2,2,NaT,1,1,0,0,1,...,1,1,1,1,-1.000000,-1.000000,-1.0,-1.0,-1.000000,Insufficient Data
6,000A005BA0BED3EA,89,2,2,NaT,0,0,0,0,0,...,0,0,0,0,-1.000000,-1.000000,-1.0,-1.0,-1.000000,Insufficient Data
7,00108066CA1FACCE,43,1,1,NaT,1,1,0,0,0,...,1,1,0,0,0.099256,0.099256,1.0,2.0,0.100672,High Risk
8,0011714C14B52EEB,68,2,1,NaT,0,0,0,1,0,...,1,0,0,0,0.208333,0.208333,1.0,2.0,0.400242,Medium Risk
9,0011CB1FE23E91AF,71,1,1,NaT,1,1,1,1,0,...,1,0,0,0,0.152284,0.152284,1.0,2.0,0.254524,High Risk



Saved patient_features.csv successfully!


In [23]:
print("Patients in patient_pdc (should have a real tier):", len(patient_pdc))
print("Patients with a real risk tier in final merged file:", 
      patient_features[patient_features['risk_tier'] != 'Insufficient Data'].shape[0])

Patients in patient_pdc (should have a real tier): 20224
Patients with a real risk tier in final merged file: 19444


In [24]:
# Find patients that ARE in patient_pdc but NOT in bene_clean
pdc_patients = set(patient_pdc['DESYNPUF_ID'])
bene_clean_patients = set(bene_clean['DESYNPUF_ID'])

missing_patients = pdc_patients - bene_clean_patients
print("Patients in patient_pdc but missing from bene_clean:", len(missing_patients))

# Double check: do these missing patients exist in the ORIGINAL (unfiltered) beneficiary_2008 file?
missing_but_in_raw = missing_patients & set(beneficiary_2008['DESYNPUF_ID'])
print("Of those, how many exist in the raw 2008 file (just filtered out by Part D coverage):", len(missing_but_in_raw))

# Check their PLAN_CVRG_MOS_NUM to confirm this is indeed the cause
check = beneficiary_2008[beneficiary_2008['DESYNPUF_ID'].isin(missing_patients)]
print("\nPLAN_CVRG_MOS_NUM distribution for these missing patients:")
print(check['PLAN_CVRG_MOS_NUM'].value_counts().sort_index())

Patients in patient_pdc but missing from bene_clean: 780
Of those, how many exist in the raw 2008 file (just filtered out by Part D coverage): 780

PLAN_CVRG_MOS_NUM distribution for these missing patients:
PLAN_CVRG_MOS_NUM
0      67
1       5
2       6
3      22
4      14
5      19
6      25
7      27
8      76
9      95
10    139
11    285
Name: count, dtype: int64


Finalizing a model ready verison

In [25]:
# Save a model-ready version: only patients with a real risk tier
patient_features_model_ready = patient_features[patient_features['risk_tier'] != 'Insufficient Data'].copy()

print("Model-ready shape:", patient_features_model_ready.shape)
print(patient_features_model_ready['risk_tier'].value_counts())

patient_features_model_ready.to_csv("dataSets/patient_features_model_ready.csv", index=False)
print("\nSaved patient_features_model_ready.csv")

Model-ready shape: (19444, 22)
risk_tier
Low Risk       6610
High Risk      6420
Medium Risk    6414
Name: count, dtype: int64

Saved patient_features_model_ready.csv


Building a V2 data

In [26]:
# --- Check viability of early/late time-split approach ---

cutoff_date = pd.Timestamp('2009-07-01')

# Use the maintenance-filtered PDE data we already built (pde_pdc9)
early = pde_pdc9[pde_pdc9['SRVC_DT'] < cutoff_date]
late = pde_pdc9[pde_pdc9['SRVC_DT'] >= cutoff_date]

early_patients = set(early['DESYNPUF_ID'])
late_patients = set(late['DESYNPUF_ID'])

print("Patients with maintenance-drug fills BEFORE cutoff:", len(early_patients))
print("Patients with maintenance-drug fills AFTER cutoff:", len(late_patients))

both_sides = early_patients & late_patients
print("Patients with fills on BOTH sides of cutoff:", len(both_sides))

# Check: of patients with fills on both sides, how many have enough fills 
# on each side to actually compute meaningful features/PDC (2+ fills each side)?
early_counts = early.groupby('DESYNPUF_ID').size()
late_counts = late.groupby('DESYNPUF_ID').size()

early_2plus = set(early_counts[early_counts >= 2].index)
late_2plus = set(late_counts[late_counts >= 2].index)

viable_patients = early_2plus & late_2plus
print("\nPatients with 2+ fills on EACH side of cutoff (viable for this approach):", len(viable_patients))

Patients with maintenance-drug fills BEFORE cutoff: 17286
Patients with maintenance-drug fills AFTER cutoff: 14842
Patients with fills on BOTH sides of cutoff: 11904

Patients with 2+ fills on EACH side of cutoff (viable for this approach): 2790


In [27]:
# --- Build additional behavioral features (Option 2) ---

behavior_features = pde_pdc9.groupby('DESYNPUF_ID').agg(
    total_fill_events=('SRVC_DT', 'count'),          # how many times they filled anything (maintenance drugs)
    avg_days_supply=('DAYS_SUPLY_NUM', 'mean'),        # typical prescription length
    avg_cost_burden=('PTNT_PAY_AMT', 'mean'),          # what they typically pay out of pocket
    total_cost_burden=('PTNT_PAY_AMT', 'sum'),         # total out-of-pocket cost across the period
    avg_total_rx_cost=('TOT_RX_CST_AMT', 'mean'),      # average gross drug cost (proxy for drug expense/complexity)
).reset_index()

print("Behavior features shape:", behavior_features.shape)
print(behavior_features.describe())
display(behavior_features.head())

Behavior features shape: (20224, 6)
       total_fill_events  avg_days_supply  avg_cost_burden  total_cost_burden  \
count       20224.000000     20224.000000     20224.000000       20224.000000   
mean            2.877324        37.263621        15.132098          44.900119   
std             1.538069        14.305973        19.887966          61.553508   
min             2.000000         0.000000         0.000000           0.000000   
25%             2.000000        30.000000         1.666667          10.000000   
50%             2.000000        30.000000         7.500000          20.000000   
75%             4.000000        45.000000        20.000000          60.000000   
max            17.000000        90.000000       170.000000         650.000000   

       avg_total_rx_cost  
count       20224.000000  
mean           83.912928  
std            80.258488  
min             0.000000  
25%            20.000000  
50%            65.000000  
75%           115.000000  
max           570.

,DESYNPUF_ID,total_fill_events,avg_days_supply,avg_cost_burden,total_cost_burden,avg_total_rx_cost
0,00013D2EFD8E45D1,5,42.0,82.000000,410.0,60.000000
1,0001FDD721E223DC,6,50.0,18.333333,110.0,33.333333
2,00036A21B65B0206,2,30.0,5.000000,10.0,5.000000
3,00108066CA1FACCE,2,20.0,15.000000,30.0,75.000000
4,0011714C14B52EEB,2,30.0,0.000000,0.0,110.000000


In [28]:
# --- Merge new behavior features into the model-ready dataset ---

patient_features_model_ready = pd.read_csv("dataSets/patient_features_model_ready.csv")

patient_features_v2 = patient_features_model_ready.merge(
    behavior_features, on='DESYNPUF_ID', how='left'
)

print("Merged shape:", patient_features_v2.shape)

# Sanity check: should be no new NaNs introduced, since every patient in 
# patient_features_model_ready already came from pdc_calc9 (same source as pde_pdc9)
print("\nMissing values check:")
print(patient_features_v2.isna().sum())

# --- Clearly separate columns by role, for your team's clarity ---
leakage_risk_cols = ['avg_pdc', 'min_pdc', 'pdc_percentile']   # DO NOT use as model input -- these define the label
safe_predictor_cols = ['AGE', 'BENE_SEX_IDENT_CD', 'BENE_RACE_CD'] + \
    [c for c in patient_features_v2.columns if c.startswith('SP_')] + \
    ['total_fill_events', 'avg_days_supply', 'avg_cost_burden', 'total_cost_burden', 'avg_total_rx_cost', 'num_maintenance_drugs']
label_col = 'risk_tier'

print("\nSafe predictor columns for modeling:")
print(safe_predictor_cols)


Merged shape: (19444, 27)

Missing values check:
DESYNPUF_ID                  0
AGE                          0
BENE_SEX_IDENT_CD            0
BENE_RACE_CD                 0
BENE_DEATH_DT            19429
SP_ALZHDMTA                  0
SP_CHF                       0
SP_CHRNKIDN                  0
SP_CNCR                      0
SP_COPD                      0
SP_DEPRESSN                  0
SP_DIABETES                  0
SP_ISCHMCHT                  0
SP_OSTEOPRS                  0
SP_RA_OA                     0
SP_STRKETIA                  0
avg_pdc                      0
min_pdc                      0
num_maintenance_drugs        0
total_fills                  0
pdc_percentile               0
risk_tier                    0
total_fill_events            0
avg_days_supply              0
avg_cost_burden              0
total_cost_burden            0
avg_total_rx_cost            0
dtype: int64

Safe predictor columns for modeling:
['AGE', 'BENE_SEX_IDENT_CD', 'BENE_RACE_CD', 'SP_ALZHDMTA', 'SP

In [29]:
print("Deceased patients in our v2 dataset (using real BENE_DEATH_DT):", patient_features_v2['BENE_DEATH_DT'].notna().sum())
print("Out of total:", len(patient_features_v2))

Deceased patients in our v2 dataset (using real BENE_DEATH_DT): 15
Out of total: 19444


In [30]:
# Drop deceased patients (tiny group, not meaningful as a feature at this sample size)
patient_features_v2 = patient_features_v2[patient_features_v2['BENE_DEATH_DT'].isna()].copy()
patient_features_v2 = patient_features_v2.drop(columns=['BENE_DEATH_DT'])

print("Final modeling dataset shape:", patient_features_v2.shape)
print(patient_features_v2['risk_tier'].value_counts())

Final modeling dataset shape: (19429, 26)
risk_tier
Low Risk       6598
High Risk      6420
Medium Risk    6411
Name: count, dtype: int64


In [31]:
patient_features_v2.to_csv("dataSets/patient_features_v2.csv", index=False)
print("\nSaved patient_features_v2.csv")


Saved patient_features_v2.csv
